# Diversity & Novelty Evaluation — Synthetic Output

Evaluasi BLEU (Self-BLEU & BLEU vs seed), Jaccard bigram, dan vocabulary expansion dari synthetic CSV Javanese.

## Metric yang dihitung

| Metric | Scope | Interpretasi |
|--------|-------|--------------|
| **Self-BLEU** | within synthetic (per label group) | Lower = more diverse |
| **BLEU vs seed** | synthetic vs original NusaX train | Lower = more novel (less paraphrasing) |
| **Avg Jaccard vs seed** | synthetic vs original NusaX train | Lower = less surface overlap |
| **Avg Pairwise Jaccard** | pairwise within label group | Lower = more diverse |
| **Vocab expansion rate** | synthetic words not in seed | Higher = more new vocabulary introduced |

## Interpretasi BLEU untuk case kita

BLEU awalnya untuk machine translation (candidate vs reference), tapi di text generation sering dipakai untuk:
- **Self-BLEU (Zhu et al. 2018 Texygen)**: diversity metric
- **BLEU vs original** (inspired by Self-Instruct Wang 2022): novelty metric

**Rentang interpretasi (rule of thumb)**:
- BLEU/Self-BLEU < 0.10 → sangat diverse (output hampir tidak overlap n-gram)
- 0.10–0.25 → healthy diversity (augmentation OK)
- 0.25–0.40 → moderate overlap (mulai kurang diverse)
- > 0.40 → high overlap (banyak template repetition atau paraphrasing)

**Jaccard bigram** adalah komplementer — fast pairwise surface overlap. Kita sudah pakai 0.5 sebagai filter threshold di pipeline.

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from itertools import combinations

pd.set_option('display.max_colwidth', 120)

ROOT = Path('y:/Michh/Python/Projects/MAGenerator')
SEED_PATH = ROOT / 'data/nusax_senti/jav/train.csv'
SYN_BEFORE = ROOT / 'outputs/synthetic/jav/synthetic_before.csv'
SYN_TEST   = ROOT / 'outputs/synthetic/jav/synthetic_test.csv'

## Load data

In [ ]:
seed_df = pd.read_csv(SEED_PATH)

# synthetic_before has trailing comma (old format) → read with 4 cols, drop extra
syn_before = pd.read_csv(SYN_BEFORE, names=['id','text','label','_extra'], header=0)
if '_extra' in syn_before.columns:
    syn_before = syn_before.drop(columns=['_extra'])

syn_test = pd.read_csv(SYN_TEST)

print(f'Seed (original train):  {len(seed_df)} rows')
print(f'  Label dist: {seed_df["label"].value_counts().to_dict()}')
print(f'Synthetic BEFORE:       {len(syn_before)} rows')
print(f'  Label dist: {syn_before["label"].value_counts().to_dict()}')
print(f'Synthetic TEST (latest):{len(syn_test)} rows')
print(f'  Label dist: {syn_test["label"].value_counts().to_dict()}')

In [ ]:
# Word count distributions per label
def word_stats(df, name):
    df = df.copy()
    df['wc'] = df['text'].str.split().str.len()
    print(f'\n=== {name} word count per label ===')
    print(df.groupby('label')['wc'].agg(['mean','std','min','max']).round(1))

word_stats(seed_df, 'Seed (original train)')
word_stats(syn_before, 'Synthetic BEFORE')
word_stats(syn_test, 'Synthetic TEST')

## Metric functions

In [ ]:
smoothie = SmoothingFunction().method1

def tokenize(text):
    return text.lower().split()

def vocab_set(texts):
    v = set()
    for t in texts:
        v |= set(re.findall(r'\b\w+\b', t.lower()))
    return v

def jaccard_bigram(s1, s2):
    w1, w2 = tokenize(s1), tokenize(s2)
    bg1 = set(zip(w1, w1[1:]))
    bg2 = set(zip(w2, w2[1:]))
    if not bg1 or not bg2:
        return 0.0
    return len(bg1 & bg2) / len(bg1 | bg2)

def self_bleu(sentences, weights=(0.25, 0.25, 0.25, 0.25)):
    """Self-BLEU (Zhu et al. 2018 Texygen)."""
    if len(sentences) < 2:
        return 0.0
    tokens = [tokenize(s) for s in sentences]
    scores = []
    for i in range(len(tokens)):
        hyp = tokens[i]
        refs = [tokens[j] for j in range(len(tokens)) if j != i]
        score = sentence_bleu(refs, hyp, weights=weights, smoothing_function=smoothie)
        scores.append(score)
    return float(np.mean(scores))

def bleu_vs_reference(hypotheses, references, weights=(0.25, 0.25, 0.25, 0.25)):
    """Avg BLEU of each hypothesis against the reference corpus."""
    if not hypotheses or not references:
        return 0.0
    hyp_tokens = [tokenize(h) for h in hypotheses]
    ref_tokens = [tokenize(r) for r in references]
    scores = [sentence_bleu(ref_tokens, h, weights=weights, smoothing_function=smoothie) for h in hyp_tokens]
    return float(np.mean(scores))

def avg_jaccard_vs_reference(hypotheses, references):
    """Avg MAX Jaccard of each hypothesis vs reference corpus."""
    scores = []
    for h in hypotheses:
        max_j = max((jaccard_bigram(h, r) for r in references), default=0.0)
        scores.append(max_j)
    return float(np.mean(scores))

def avg_pairwise_jaccard(sentences, max_pairs=5000):
    """Avg pairwise Jaccard among sentences (diversity)."""
    if len(sentences) < 2:
        return 0.0
    pairs = list(combinations(sentences, 2))
    if len(pairs) > max_pairs:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(pairs), size=max_pairs, replace=False)
        pairs = [pairs[i] for i in idx]
    scores = [jaccard_bigram(a, b) for a, b in pairs]
    return float(np.mean(scores))

## Compute metrics per label, per dataset

In [ ]:
def evaluate(df, name, seed_df):
    rows = []
    for label in ['negative', 'neutral', 'positive']:
        syn_texts  = df[df['label']==label]['text'].dropna().tolist()
        seed_texts = seed_df[seed_df['label']==label]['text'].dropna().tolist()
        if not syn_texts:
            continue
        rows.append({
            'dataset': name,
            'label': label,
            'n_synth': len(syn_texts),
            'n_seed': len(seed_texts),
            'Self-BLEU': round(self_bleu(syn_texts), 4),
            'BLEU vs seed': round(bleu_vs_reference(syn_texts, seed_texts), 4),
            'Avg Jaccard vs seed': round(avg_jaccard_vs_reference(syn_texts, seed_texts), 4),
            'Avg Pairwise Jaccard': round(avg_pairwise_jaccard(syn_texts), 4),
        })
    return rows

all_results = []
all_results.extend(evaluate(syn_before, 'synthetic_before', seed_df))
all_results.extend(evaluate(syn_test,   'synthetic_test',   seed_df))

result_df = pd.DataFrame(all_results)
result_df

## Aggregate per dataset (macro avg over labels)

In [ ]:
agg = result_df.groupby('dataset')[['Self-BLEU', 'BLEU vs seed', 'Avg Jaccard vs seed', 'Avg Pairwise Jaccard']].mean().round(4)
agg

## Interpretation helper

In [ ]:
def interpret_bleu(score):
    if score < 0.10: return 'very diverse / very novel'
    elif score < 0.25: return 'healthy diversity / novelty'
    elif score < 0.40: return 'moderate overlap'
    else: return 'high overlap - concerning'

def interpret_jaccard_vs_seed(score):
    if score < 0.10: return 'very novel surface'
    elif score < 0.25: return 'healthy surface novelty'
    elif score < 0.50: return 'moderate surface overlap'
    else: return 'high surface overlap - possible paraphrase'

print('=== Interpretation per dataset (macro avg) ===')
for ds in agg.index:
    row = agg.loc[ds]
    print(f'\n{ds}:')
    print(f"  Self-BLEU           = {row['Self-BLEU']:.4f} -> {interpret_bleu(row['Self-BLEU'])}")
    print(f"  BLEU vs seed        = {row['BLEU vs seed']:.4f} -> {interpret_bleu(row['BLEU vs seed'])}")
    print(f"  Avg Jaccard vs seed = {row['Avg Jaccard vs seed']:.4f} -> {interpret_jaccard_vs_seed(row['Avg Jaccard vs seed'])}")
    print(f"  Avg Pairwise Jaccard= {row['Avg Pairwise Jaccard']:.4f}")

## Vocabulary expansion check

Seberapa banyak kata di synthetic yang TIDAK ada di seed train? Ini yang justify augmentation goal (mengisi OOV gap 50% di test set).

In [ ]:
seed_vocab = vocab_set(seed_df['text'].dropna().tolist())
print(f'Seed train vocab: {len(seed_vocab)} unique words\n')

vocab_rows = []
for name, df in [('synthetic_before', syn_before), ('synthetic_test', syn_test)]:
    syn_vocab = vocab_set(df['text'].dropna().tolist())
    new_words = syn_vocab - seed_vocab
    shared = syn_vocab & seed_vocab
    vocab_rows.append({
        'dataset': name,
        'synth_vocab': len(syn_vocab),
        'shared_w_seed': len(shared),
        'shared_pct': round(100*len(shared)/len(syn_vocab), 1),
        'new_words': len(new_words),
        'new_pct': round(100*len(new_words)/len(syn_vocab), 1),
    })
vocab_df = pd.DataFrame(vocab_rows)
print(vocab_df.to_string(index=False))

# Sample new words from latest synthetic
syn_vocab_test = vocab_set(syn_test['text'].dropna().tolist())
new_words_test = list(syn_vocab_test - seed_vocab)
print(f'\nSample 20 new words in synthetic_test (not in seed):')
print(new_words_test[:20])

## Inspeksi sample yang score-nya tinggi (debugging)

In [ ]:
def inspect_high_similarity(syn_df, seed_df, label, top_n=3):
    syn_texts = syn_df[syn_df['label']==label]['text'].dropna().tolist()
    seed_texts = seed_df[seed_df['label']==label]['text'].dropna().tolist()
    pairs = []
    for h in syn_texts:
        best_j = 0.0
        best_ref = ''
        for r in seed_texts:
            j = jaccard_bigram(h, r)
            if j > best_j:
                best_j, best_ref = j, r
        pairs.append((best_j, h, best_ref))
    pairs.sort(reverse=True)
    print(f'\n=== TOP {top_n} HIGHEST Jaccard vs seed - {label} ===')
    for j, h, r in pairs[:top_n]:
        print(f'\nJaccard={j:.3f}')
        print(f'  SYNTH: {h[:150]}')
        print(f'  SEED:  {r[:150]}')

inspect_high_similarity(syn_test, seed_df, 'negative', top_n=3)
inspect_high_similarity(syn_test, seed_df, 'positive', top_n=3)

## Diskusi: sweet spot BLEU untuk case ini

Data empiris menunjukkan **~50% kata di test set NusaX TIDAK ADA di train** (per analisis vocab). Augmentation goal = vocabulary expansion.

Untuk goal ini, BLEU rendah (0.05-0.15) terhadap seed adalah **aligned**, bukan terlalu rendah. Domain preservation dijamin via Contextualizer (analisis domain) + Generator prompt (match style). Linguistic quality dijamin via LV validator.

### Philosophy guide

| Philosophy | BLEU vs seed | Use case |
|------------|--------------|---------|
| **Vocabulary expansion** | 0.05-0.15 | Training set vocab-limited, test OOV tinggi (our case) |
| Style matching | 0.15-0.25 | Need strict adherence ke source style |
| Paraphrase-like | 0.25-0.40 | Rarely augmentation |
| Too close (paraphrase) | >0.40 | Rejected |

### Kesimpulan untuk thesis

Gunakan **keduanya** karena role-nya berbeda:
- **Jaccard sebagai runtime filter** (`collect_node` threshold 0.5): cepat, per-pair, standard di dedup literature
- **BLEU sebagai post-hoc evaluation**: corpus-level aggregate, standard untuk diversity/novelty reporting

Sama-sama surface-level (n-gram based) — cocok untuk low-resource languages di mana semantic metrics (BERTScore) unreliable.